In [0]:
from pyspark.sql.functions import (
    col,
    avg,
    min,
    max,
    sum,
    count,
    round
)

# Read silver weather table
df = spark.table("portfolio.weather_api_pipeline.silver_weather_sao_paulo")

# Create monthly analytical table
monthly_df = (
    df
    .groupBy("year", "month", "month_name", "city")
    .agg(
        count("*").alias("total_days"),
        round(avg("temperature_mean_celsius"), 2).alias("avg_temperature_celsius"),
        round(avg("temperature_max_celsius"), 2).alias("avg_max_temperature_celsius"),
        round(avg("temperature_min_celsius"), 2).alias("avg_min_temperature_celsius"),
        round(max("temperature_max_celsius"), 2).alias("highest_temperature_celsius"),
        round(min("temperature_min_celsius"), 2).alias("lowest_temperature_celsius"),
        round(sum("precipitation_mm"), 2).alias("total_precipitation_mm"),
        sum("rain_flag").alias("rainy_days"),
        round(avg("windspeed_max_kmh"), 2).alias("avg_windspeed_max_kmh"),
        round(avg("temperature_range_celsius"), 2).alias("avg_temperature_range_celsius")
    )
    .withColumn(
        "rainy_days_rate",
        round(col("rainy_days") / col("total_days"), 4)
    )
    .orderBy("year", "month")
)

# Save as gold Delta table
monthly_df.write.mode("overwrite").saveAsTable(
    "portfolio.weather_api_pipeline.gold_monthly_weather_sao_paulo"
)

display(monthly_df)